# NB-07 — End-to-End Inference Pipeline

**Goal:** Use `MultimodalQAPipeline` as a single entry point for real multimodal QA.

Supports: file paths, URLs, base64, PIL images, multi-turn history, text-only fallback, and optional streaming.

---

In [1]:
import sys
from pathlib import Path

sys.path.insert(0, str(Path("..").resolve()))

import time
import base64
from io import BytesIO

import torch
from dotenv import load_dotenv
from PIL import Image

load_dotenv(Path("..") / ".env")

from src.pipeline.multimodal_qa import ImageLoadError, MultimodalQAPipeline

DEVICE = "mps" if torch.backends.mps.is_available() else "cuda" if torch.cuda.is_available() else "cpu"
CONFIG = Path("..") / "config" / "model_config.yaml"
DEVICE = "cpu"
print(f"Device: {DEVICE}")

Device: cpu


## 1. Load the full pipeline

First run downloads CLIP + Qwen2-VL-2B (~4GB). On Apple Silicon, `encoder_on_cpu=True` saves MPS memory for the LLM.

In [2]:
print("Loading MultimodalQAPipeline...")
pipeline = MultimodalQAPipeline(
    CONFIG,
    device=DEVICE,
    encoder_on_cpu=(DEVICE == "mps"),
)
print(f"Max new tokens: {pipeline.max_new_tokens}")

2026-05-26 16:54:24.015 | INFO     | src.pipeline.multimodal_qa:__init__:75 - Initializing MultimodalQAPipeline from ../config/model_config.yaml


Loading MultimodalQAPipeline...


Loading weights:   0%|          | 0/590 [00:00<?, ?it/s]

2026-05-26 16:54:29.405 | INFO     | src.llm.backbone:__init__:114 - Loading LLM: Qwen/Qwen2-VL-2B-Instruct


Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/729 [00:00<?, ?it/s]

2026-05-26 16:54:47.115 | INFO     | src.llm.backbone:__init__:120 - MultimodalLLM ready | device=cpu | llm_hidden=1536 | encoder_on_cpu=False


Max new tokens: 512


## 2. Demo — single image QA (URL input)

In [4]:
IMG_URL = (
    "https://images.unsplash.com/photo-1518717758536-85ae29035b6d?auto=format&fit=crop&w=224&q=80"
)
question = "What animal is in this image? Answer in one short sentence."

t0 = time.perf_counter()
answer = pipeline.answer(question, image=IMG_URL)
latency = time.perf_counter() - t0

print(f"Latency: {latency:.2f}s")
print(f"Answer: {answer}")

2026-05-26 16:55:30.142 | INFO     | src.pipeline.multimodal_qa:answer:135 - answer | has_image=True | history_turns=0 | stream=False | native=False | rag=False
2026-05-26 16:55:33.420 | INFO     | src.llm.backbone:generate:330 - Custom generate | visual_tokens=256 text_tokens=13 total=269
2026-05-26 16:56:47.941 | INFO     | src.llm.backbone:generate:353 - Generation done | latency=77.80s | ~0.1 tok/s


Latency: 78.09s
Answer: A dog.


## 3. Demo — multi-turn conversation (same image)

History is a list of `{"role": "user"|"assistant", "content": str}` turns.

In [5]:
history: list[dict[str, str]] = []
rounds = [
    "What is the main subject?",
    "What colors do you notice?",
    "Would this be a good pet photo? Why?",
]

for i, q in enumerate(rounds, 1):
    reply = pipeline.answer(q, image=IMG_URL, history=history, max_new_tokens=128)
    print(f"--- Turn {i} ---")
    print(f"User: {q}")
    print(f"Assistant: {reply}\n")
    history.append({"role": "user", "content": q})
    history.append({"role": "assistant", "content": reply})

2026-05-26 16:56:55.820 | INFO     | src.pipeline.multimodal_qa:answer:135 - answer | has_image=True | history_turns=0 | stream=False | native=False | rag=False
2026-05-26 16:56:59.316 | INFO     | src.llm.backbone:generate:330 - Custom generate | visual_tokens=256 text_tokens=6 total=262
2026-05-26 16:57:22.156 | INFO     | src.llm.backbone:generate:353 - Generation done | latency=26.33s | ~4.9 tok/s
2026-05-26 16:57:22.268 | INFO     | src.pipeline.multimodal_qa:answer:135 - answer | has_image=True | history_turns=2 | stream=False | native=False | rag=False


--- Turn 1 ---
User: What is the main subject?
Assistant: The main subject of the passage is the concept of "The Law of Attraction" and its relationship with the law of attraction. The Law of Attraction is a belief that people are attracted to things that align with their desires and beliefs. The Law of Attraction is often used in the context of the law of attraction to describe the idea that people are attracted to things that align with their desires and beliefs. The Law of Attraction is often used in the context of the law of attraction to describe the idea that people are attracted to things that align with their desires and beliefs. The Law of Attraction is often used in the context of



2026-05-26 16:57:25.789 | INFO     | src.llm.backbone:generate:330 - Custom generate | visual_tokens=256 text_tokens=149 total=405
2026-05-26 16:57:46.071 | INFO     | src.llm.backbone:generate:353 - Generation done | latency=23.80s | ~5.4 tok/s
2026-05-26 16:57:46.179 | INFO     | src.pipeline.multimodal_qa:answer:135 - answer | has_image=True | history_turns=4 | stream=False | native=False | rag=False


--- Turn 2 ---
User: What colors do you notice?
Assistant: The colors that I notice are red, white, and blue. The colors that I notice are red, white, and blue. The colors that I notice are red, white, and blue. The colors that I notice are red, white, and blue. The colors that I notice are red, white, and blue. The colors that I notice are red, white, and blue. The colors that I notice are red, white, and blue. The colors that I notice are red, white, and blue. The colors that I notice are red, white, and blue. The colors that I notice are red, white, and



2026-05-26 16:57:46.771 | INFO     | src.llm.backbone:generate:330 - Custom generate | visual_tokens=256 text_tokens=292 total=548
2026-05-26 16:58:38.436 | INFO     | src.llm.backbone:generate:353 - Generation done | latency=52.26s | ~2.4 tok/s


--- Turn 3 ---
User: Would this be a good pet photo? Why?
Assistant: This would be a good pet photo because it would be a good pet photo because it would be a good pet photo because it would be a good pet photo because it would be a good pet photo because it would be a good pet photo because it would be a good pet photo because it would be a good pet photo because it would be a good pet photo because it would be a good pet photo because it would be a good pet photo because it would be a good pet photo because it would be a good pet photo because it would be a good pet photo because it would be a good pet photo because it would be a good pet photo because



## 4. Demo — text-only (no image)

The custom path still runs through the language model with text embeddings only.

In [6]:
text_only = pipeline.answer(
    "Explain contrastive learning in one paragraph.",
    image=None,
    max_new_tokens=200,
)
print(text_only)

2026-05-26 16:59:20.701 | INFO     | src.pipeline.multimodal_qa:answer:135 - answer | has_image=False | history_turns=0 | stream=False | native=False | rag=False
2026-05-26 16:59:20.757 | INFO     | src.llm.backbone:generate:330 - Custom generate | visual_tokens=0 text_tokens=9 total=9
2026-05-26 16:59:42.407 | INFO     | src.llm.backbone:generate:353 - Generation done | latency=21.70s | ~6.6 tok/s


aims to learn representations of different languages by comparing them to each other. It works by training a model to recognize patterns and similarities between sentences from different languages, rather than learning a single, universal representation. This approach is particularly useful for tasks such as translation, where the goal is to translate one language into another. Contrastive learning has been shown to be effective in improving the performance of machine translation systems. It also has applications in other areas such as sentiment analysis and language modeling. However, it can be computationally expensive and may not always be able to capture the nuances of language. Overall, contrastive learning is a powerful tool for learning language representations, but it requires careful consideration of its limitations and potential applications.


## 5. Input types — path, PIL, base64

In [7]:
pil_img = pipeline._load_image(IMG_URL)
cache_path = Path("..") / "data" / "nb07_sample.jpg"
cache_path.parent.mkdir(parents=True, exist_ok=True)
pil_img.save(cache_path)

buf = BytesIO()
pil_img.save(buf, format="JPEG")
b64 = base64.b64encode(buf.getvalue()).decode("ascii")

for label, src in [("path", str(cache_path)), ("PIL", pil_img), ("base64", b64)]:
    loaded = pipeline._load_image(src)
    print(f"{label}: {loaded.size} {loaded.mode}")

path: (224, 149) RGB
PIL: (224, 149) RGB
base64: (224, 149) RGB


## 6. Edge cases and error handling

In [8]:
def try_answer(label: str, **kwargs) -> None:
    try:
        out = pipeline.answer(**kwargs, max_new_tokens=64)
        print(f"[{label}] OK — {str(out)[:120]}...")
    except (ImageLoadError, ValueError) as exc:
        print(f"[{label}] Expected error: {exc}")

try_answer("bad path", question="What?", image="/tmp/not_a_real_image.jpg")
try_answer("empty question", question="   ", image=None)

long_q = "What do you see? " + "Please be detailed. " * 80
try_answer("very long question", question=long_q, image=IMG_URL)

2026-05-26 17:00:15.246 | INFO     | src.pipeline.multimodal_qa:answer:135 - answer | has_image=True | history_turns=0 | stream=False | native=False | rag=False


[bad path] Expected error: Failed to load image: Not a valid file path, URL, or base64 image: '/tmp/not_a_real_image.jpg'
[empty question] Expected error: question must be a non-empty string


2026-05-26 17:00:18.234 | INFO     | src.llm.backbone:generate:330 - Custom generate | visual_tokens=256 text_tokens=325 total=581
2026-05-26 17:00:27.481 | INFO     | src.llm.backbone:generate:353 - Generation done | latency=12.23s | ~5.2 tok/s


[very long question] OK — Please be detailed. Please be detailed. Please be detailed. Please be detailed. Please be detailed. Please be detailed. ...


## 7. Native baseline vs custom pipeline

Compare Qwen2-VL's built-in vision tower (`use_native=True`) with our CLIP + MLP projector path.

In [9]:
q = "Describe the scene briefly."

custom = pipeline.answer(q, image=IMG_URL, use_native=False, max_new_tokens=128)
native = pipeline.answer(q, image=IMG_URL, use_native=True, max_new_tokens=128)

print("Custom (CLIP+MLP):", custom)
print("Native (Qwen2-VL):", native)

2026-05-26 17:00:54.744 | INFO     | src.pipeline.multimodal_qa:answer:135 - answer | has_image=True | history_turns=0 | stream=False | native=False | rag=False
2026-05-26 17:00:56.907 | INFO     | src.llm.backbone:generate:330 - Custom generate | visual_tokens=256 text_tokens=5 total=261
2026-05-26 17:01:16.716 | INFO     | src.llm.backbone:generate:353 - Generation done | latency=21.97s | ~5.8 tok/s
2026-05-26 17:01:16.877 | INFO     | src.pipeline.multimodal_qa:answer:135 - answer | has_image=True | history_turns=0 | stream=False | native=True | rag=False
2026-05-26 17:01:59.456 | INFO     | src.llm.backbone:generate_native:421 - Native generate | latency=42.58s | new_tokens=65


Custom (CLIP+MLP): The scene is a classic example of how the law can be used to protect the environment and the environment can be used to protect the law. The scene depicts a city with a river flowing through it. The river is a symbol of the environment, and the city is a symbol of the law. The river is a symbol of the environment, and the city is a symbol of the law. The river is a symbol of the environment, and the city is a symbol of the law. The river is a symbol of the environment, and the city is a symbol of the law. The river is a symbol of the environment, and the city
Native (Qwen2-VL): The image shows a close-up of a chocolate Labrador Retriever dog. The dog has a long, pink tongue sticking out, and its eyes are wide open, looking directly at the camera. The background appears to be an outdoor setting, possibly a sidewalk or path, with a blurred background that suggests movement or activity.


## 8. Projector comparison (Linear vs MLP vs Q-Former)

Each variant reloads the model — run only when you have time/GPU memory. Q-Former uses fewer visual tokens (32 queries).

In [10]:
from omegaconf import OmegaConf
from src.llm.backbone import MultimodalLLM

COMPARE = False  # set True to run side-by-side projector comparison

if COMPARE:
    base_cfg = OmegaConf.load(CONFIG)
    pil = pipeline._load_image(IMG_URL)
    test_q = "What is in this image?"

    for ptype in ("linear", "mlp", "qformer"):
        cfg = base_cfg.copy()
        cfg.projector.type = ptype
        tmp = Path("..") / "config" / f"_nb07_{ptype}.yaml"
        OmegaConf.save(cfg, tmp)
        print(f"\n=== Projector: {ptype} ===")
        m = MultimodalLLM.from_config(tmp, device=DEVICE, encoder_on_cpu=(DEVICE == "mps"))
        ans = m.generate(pil, test_q, max_new_tokens=96)
        print(ans)
else:
    print("Set COMPARE=True to reload models for linear / mlp / qformer comparison.")

Set COMPARE=True to reload models for linear / mlp / qformer comparison.


## 9. Streaming (optional)

Custom path currently yields the full answer in one chunk; native streaming can be added in Phase 7 API work.

In [11]:
chunks = list(pipeline.answer("Name one object.", image=IMG_URL, stream=True, max_new_tokens=64))
print("Streamed chunks:", chunks)

2026-05-26 17:03:59.551 | INFO     | src.pipeline.multimodal_qa:answer:135 - answer | has_image=True | history_turns=0 | stream=True | native=False | rag=False
2026-05-26 17:03:59.553 | WARNING  | src.llm.backbone:_generate_custom_stream:368 - Streaming falls back to single-shot decode for custom path.
2026-05-26 17:04:00.630 | INFO     | src.llm.backbone:generate:330 - Custom generate | visual_tokens=256 text_tokens=4 total=260
2026-05-26 17:04:24.553 | INFO     | src.llm.backbone:generate:353 - Generation done | latency=25.00s | ~2.6 tok/s


Streamed chunks: ['The object of this essay is to discuss the importance of the role of the role of the role of the role of the role of the role of the role of the role of the role of the role of the role of the role of the role of the role of the role of the role of the role of the role']


## 10. Optional — Gradio UI

Uncomment and run if `gradio` is installed (`pip install gradio`).

In [ ]:
RUN_GRADIO = False

if RUN_GRADIO:
    import gradio as gr

    def gradio_answer(image, question, history_json):
        hist = history_json or []
        return pipeline.answer(question, image=image, history=hist, max_new_tokens=256)

    demo = gr.Interface(
        fn=gradio_answer,
        inputs=["image", "text", gr.State([])],
        outputs="text",
        title="VisionMind QA",
    )
    demo.launch()
else:
    print("Set RUN_GRADIO=True to launch a local demo.")

---

**Phase 4 complete.** Next: Phase 5 — LoRA fine-tuning (`src/llm/lora_finetune.py`, `NB-08-lora-finetuning.ipynb`).